In [ ]:
from accelforge import Spec
from accelforge.frontend.arch import (
    Arch,
    Memory,
    Action,
    Tensors,
    Compute,
    Container,
    Spatial,
    Comparison,
)
from accelforge.frontend.workload import (
    Workload,
    Einsum,
    TensorAccess,
)

from math import ceil

# =========================================================
# CONSTANTS
# =========================================================

BYTES = 8
KBs = int(BYTES * 1024**1)
MBs = int(BYTES * 1024**2)

CLOCK_HZ = 1e9
BASE_ENG_UNIT = 1  # TODO change the value

# ===================================================
# HW MODEL
# ===================================================

MAC_SPAT_UNITS = 240
NOF_ARRAYS = 8
# ===================================================
# DRAM PARAMS
# ===================================================

DRAM_BYTES_PER_ACCESS = 256
DRAM_BITS_PER_ACCESS = DRAM_BYTES_PER_ACCESS * BYTES
DRAM_SIZE_MB = 512
DRAM_SIZE_BITS = DRAM_SIZE_MB * MBs
DRAM_ENERGY = 400.0 * BASE_ENG_UNIT
DRAM_THROUGHPUT = CLOCK_HZ / 40000

# ===================================================
# SRAM PARAMS
# ===================================================

# AO SRAM
AO_ENTRY_BANK_SIZE_KB = 10000
AO_ENTRY_BANK_SIZE_BYTES =  AO_ENTRY_BANK_SIZE_KB * KBs
AO_BITS_PER_ACCESS = MAC_SPAT_UNITS * BYTES
AO_ENERGY = 4.0 * BASE_ENG_UNIT
AO_THROUGHPUT = CLOCK_HZ / 40

# W SRAM
WEIGHT_ENTRY_BANK_SIZE_KB = 10
WEIGHT_ENTRY_BANK_SIZE_BYTES =  WEIGHT_ENTRY_BANK_SIZE_KB * KBs 
WEIGHT_BITS_PER_ACCESS = MAC_SPAT_UNITS * BYTES * NOF_ARRAYS
WEIGHT_ENERGY = 4.0 * BASE_ENG_UNIT
WEIGHT_THROUGHPUT = CLOCK_HZ / 20

# ===================================================
# BUFFERs PARAMS
# ===================================================

# O-BUFFER
OUT_BUFFER_SIZE = MAC_SPAT_UNITS * BYTES * NOF_ARRAYS
OUT_BUFFER_BITS_PER_ACCESS = MAC_SPAT_UNITS * BYTES * NOF_ARRAYS
OUTPUT_REG_ENERGY = 1.0 * BASE_ENG_UNIT
OUTPUT_REG_THROUGHPUT = CLOCK_HZ / 1

# A-BUFFER
A_BUFFER_SIZE = MAC_SPAT_UNITS * BYTES
A_BUFFER_BITS_PER_ACCESS = MAC_SPAT_UNITS * BYTES
A_REG_ENERGY = 10.0 * BASE_ENG_UNIT
A_REG_THROUGHPUT = CLOCK_HZ / 20

# A-BUFFER
W_BUFFER_SIZE = MAC_SPAT_UNITS * BYTES * NOF_ARRAYS
W_BUFFER_BITS_PER_ACCESS = MAC_SPAT_UNITS * BYTES * NOF_ARRAYS
W_REG_ENERGY = 40.0 * BASE_ENG_UNIT
W_REG_THROUGHPUT = CLOCK_HZ / 10


# ===================================================
# MAC PARAMS
# ===================================================
MAC_REG_ENERGY = 1.0 * BASE_ENG_UNIT
MAC_REG_THROUGHPUT = CLOCK_HZ / 1

# =========================================================
# DRAM
# =========================================================

dram = Memory(
    name="DRAM",

    size=DRAM_SIZE_BITS,
    bits_per_action=DRAM_BITS_PER_ACCESS,

    area=0,
    leak_power=0,

    actions=[
        Action(
            name="read",
            throughput=DRAM_THROUGHPUT,
            energy=DRAM_ENERGY,
        ),
        Action(
            name="write",
            throughput=DRAM_THROUGHPUT,
            energy=DRAM_ENERGY,
        ),
    ],

    total_latency="max(a.n_calls / a.throughput for a in actions)",

    tensors=Tensors(
        # keep="Intermediates",
        # keep="Intermediatesss",
        # may_keep="Nothing",
        # may_keep="input | weight | output",

        keep="~Intermediates",
        may_keep="input | output",
    ),
)


# =========================================================
# AO
#
# TWO entries, but NOT spatial fanout.
#
# AO0 / AO1 are ping-pong storage.
# =========================================================

ao = Memory(
    name="AO",

    size=AO_ENTRY_BANK_SIZE_BYTES,

    bits_per_action=AO_BITS_PER_ACCESS,

    area=0,
    leak_power=0,

    actions=[
        Action(
            name="read",
            throughput=AO_THROUGHPUT,
            energy=AO_ENERGY,
        ),
        Action(
            name="write",
            throughput=AO_THROUGHPUT,
            energy=AO_ENERGY,
        ),
    ],

    total_latency="max(a.n_calls / a.throughput for a in actions)",

    tensors=Tensors(
        keep="input | output",
        may_keep="Intermediates",
    ),
)


# =========================================================
# WEIGHT SRAM
# =========================================================

w = Memory(
    name="W",

    size=WEIGHT_ENTRY_BANK_SIZE_BYTES,
    bits_per_action=WEIGHT_BITS_PER_ACCESS,

    area=0,
    leak_power=0,

    actions=[
        Action(
            name="read",
            throughput=W_REG_THROUGHPUT,
            energy=W_REG_ENERGY,

        ),
        Action(
            name="write",
            throughput=W_REG_THROUGHPUT,
            energy=W_REG_ENERGY,
        ),
    ],

    total_latency="max(a.n_calls / a.throughput for a in actions)",

    tensors=Tensors(
        keep="weight",
        may_keep="weight",
    ),
)

# =========================================================
# OUTPUT-STATIONARY ACCUMULATOR
# =========================================================

output_reg = Memory(
    name="OutputReg",

    # 32 x 32-bit accumulator registers
    # size=32 * 32,
    size=OUT_BUFFER_SIZE,

    bits_per_action=OUT_BUFFER_BITS_PER_ACCESS,

    area=0,
    leak_power=0,

    actions=[
        Action(
            name="read",
            throughput=OUTPUT_REG_THROUGHPUT,
            energy=OUTPUT_REG_ENERGY,
        ),
        Action(
            name="write",
            throughput=OUTPUT_REG_THROUGHPUT,
            energy=OUTPUT_REG_ENERGY,
        ),
    ],

    total_latency="max(a.n_calls / a.throughput for a in actions)",

    tensors=Tensors(
        keep="output",
        may_keep="Nothing",
        # no_refetch_from_above="output",
        # no_resend_to_below="output",
    ),
)

# =========================================================
# Activation Buffer 
# =========================================================

activation_reg = Memory(
    name="ActivationReg",
    size=A_BUFFER_SIZE,
    bits_per_action=A_BUFFER_BITS_PER_ACCESS,

    area=0,
    leak_power=0,

    actions=[
        Action(
            name="read",
            throughput=A_REG_THROUGHPUT,
            energy=A_REG_ENERGY,
        ),
        Action(
            name="write",
            throughput=A_REG_THROUGHPUT,
            energy=A_REG_ENERGY,
        ),
    ],

    total_latency="max(a.n_calls / a.throughput for a in actions)",

    tensors=Tensors(
        keep="input",
        may_keep="Nothing",
        # no_refetch_from_above="output",
        # no_resend_to_below="output",
    ),
)


# =========================================================
# Weight Buffer 
# =========================================================

weight_reg = Memory(
    name="WeightReg",

    # 32 x 32-bit accumulator registers
    # size=32 * 32,
    size=W_BUFFER_SIZE,

    bits_per_action=W_BUFFER_BITS_PER_ACCESS,

    area=0,
    leak_power=0,

    actions=[
        Action(
            name="read",
            throughput=W_REG_THROUGHPUT,
            energy=W_REG_ENERGY,
        ),
        Action(
            name="write",
            throughput=W_REG_THROUGHPUT,
            energy=W_REG_ENERGY,
        ),
    ],

    total_latency="max(a.n_calls / a.throughput for a in actions)",

    tensors=Tensors(
        keep="weight",
        may_keep="Nothing",
        # no_refetch_from_above="output",
        # no_resend_to_below="output",
    ),
)



# =========================================================
# MAC
#
# 32 MACs / cycle
# =========================================================

compute_lanes = Container(
    name="ComputeLanes",
    spatial=[
        Spatial(
            name="MACLane",
            fanout=MAC_SPAT_UNITS,

            # IMPORTANT for a 510-wide output:
            # last group only uses 30/32 lanes
            min_usage=1,

            # input changes between lanes, output changes between lanes.
            # Weight is reused by all 32 lanes.
            # reuse="weight",
            # may_reuse="weight",

            loop_bounds=[
                Comparison(
            expression="cx1 | y1 | zi | z1 | rx | ry",
            operator="==",
                    value=1,
                ),
            ],
        ),
    ],
)

mac = Compute(
    name="MAC",
    area=0,
    leak_power=0,
    actions=[
        Action(
            name="compute",
            throughput=MAC_REG_THROUGHPUT,  # one MAC per lane per cycle
            energy=MAC_REG_ENERGY,
        ), 
    ],
)

# =========================================================
# ARCH
# =========================================================

arch = Arch(
    nodes=[
        dram,
        ao,
        w,
        output_reg,
        activation_reg,
        weight_reg,
        compute_lanes,
        mac,
    ]
)

# =========================================================
# TWO CONVOLUTIONS
# =========================================================

H_INPUT = 512
W_INPUT = 512
C_INPUT = 3

KERNEL_SIZE = 3
STRIDE = 1

C1_OUTPUT = 8
C2_OUTPUT = 8

H1 = (H_INPUT - KERNEL_SIZE) // STRIDE + 1
W1 = (W_INPUT - KERNEL_SIZE) // STRIDE + 1

H2 = (H1 - KERNEL_SIZE) // STRIDE + 1
W2 = (W1 - KERNEL_SIZE) // STRIDE + 1

iteration_space_shape = {
    # Conv1
    "x1": f"{ceil((W1)/MAC_SPAT_UNITS)-1} <= x1 < {ceil((W1)/MAC_SPAT_UNITS)}",       # ceil(254 / 32)
    "y1": f"{(H1 - 1)} <= y1 < {(H1)}",       # ceil(254 / 32)

    # Conv2
    "x2": f"{ceil((W2)/MAC_SPAT_UNITS)-1} <= x2 < {ceil((W2)/MAC_SPAT_UNITS)}",       # ceil(254 / 32)
    "y2": f"{(H2 - 1)} <= y2 < {(H2)}",       # ceil(254 / 32)

}

LANES = 32

# iteration_space_shape = {
    # "cx1": f"0 <= cx1 < {ceil(W1 / LANES)}",
    # "lx1": f"0 <= lx1 < {LANES}",
    # "y1":  f"0 <= y1 < {H1}",

    # "zi":  f"0 <= zi < {CIN1}",
    # "z1":  f"0 <= z1 < {COUT1}",
    # "rx":  "0 <= rx < 3",
    # "ry":  "0 <= ry < 3",
# }

rank_sizes = {
    "N": 1,

    # Original input
    "XI": H_INPUT,
    "YI": W_INPUT,
    "ZI": C_INPUT,

    # Conv1 output / Conv2 input
    "X1": H1,
    "Y1": W1,
    "Z1": C1_OUTPUT,

    # Conv2 output
    "X2": H2,
    "Y2": W2,
    "Z2": C2_OUTPUT,

    # Kernel
    "RX": KERNEL_SIZE,
    "RY": KERNEL_SIZE,
}

workload = Workload(
    bits_per_value={
        "All": 8,
    },

    iteration_space_shape=iteration_space_shape,

    rank_sizes=rank_sizes,

    einsums=[

        # =================================================
        # CONV 1
        #
        # 512x512x3
        #       |
        #      3x3
        #       v
        # 510x510x8
        # =================================================
        Einsum(
            name="Conv1",
            tensor_accesses=[
                TensorAccess(
                    name="A",
                    projection={
                        "N": "n",
                        "ZI": "z0",
                        "YI": "y1 + ry",
                        "XI": "x1 + rx",
                    },
                    output=False,
                ),
        
                TensorAccess(
                    name="W1",
                    projection={
                        "Z1": "z1",
                        "RX": "rx",
                        "RY": "ry",
                        "ZI": "z0",
                    },
                    output=False,
                ),
        
                TensorAccess(
                    name="O1",
                    projection={
                        "N": "n",
                        "Z1": "z1",
                        "Y1": "y1",
                        "X1": "x1",
                    },
                    output=True,
                ),
            ],
        
            renames={
                "input": "A",
                "weight": "W1",
                "output": "O1",
            },
        ),
        
        Einsum(
            name="Conv2",
            tensor_accesses=[
                TensorAccess(
                    name="O1",
                    projection={
                        "N": "n",
                        "Z1": "z1",
                        "Y1": "y2 + ry2",
                        "X1": "x2 + rx2",
                    },
                    output=False,
                ),
        
                TensorAccess(
                    name="W2",
                    projection={
                        "Z2": "z2",
                        "RX": "rx2",
                        "RY": "ry2",
                        "Z1": "z1",
                    },
                    output=False,
                ),
        
                TensorAccess(
                    name="O2",
                    projection={
                        "N": "n",
                        "Z2": "z2",
                        "Y2": "y2",
                        "X2": "x2",
                    },
                    output=True,
                ),
            ],
        
            renames={
                "input": "O1",
                "weight": "W2",
                "output": "O2",
            },
        ),
    ],
)

from accelforge.frontend.mapping import (
    Mapping,
    Storage,
    Temporal,
    Spatial,
    Compute,
    Sequential,
    Nested,
)


# ============================================================
# HAND-WRITTEN MAPPING
#
# Loop order:
#
# Conv1:
#   n
#   y1
#   x1 tile of MAC_SPAT_UNITS
#   z1 tile of NOF_ARRAYS
#   z0
#   ry
#   rx
#   spatial x1 across MAC_SPAT_UNITS
#
# Conv2:
#   n
#   y2
#   x2 tile of MAC_SPAT_UNITS
#   z2 tile of NOF_ARRAYS
#   z1
#   ry2
#   rx2
#   spatial x2 across MAC_SPAT_UNITS
#
# iteration_space_shape stays EXACTLY as you defined it.
# ============================================================

hand_mapping = Mapping(
    nodes=[

        # ====================================================
        # DRAM
        # ====================================================
        Storage(
            component="DRAM",
            tensors=[
                "A",
                "W1",
                "O1",
                "W2",
                "O2",
            ],
        ),

        # ====================================================
        # Execute Conv1 then Conv2
        # ====================================================
        Sequential(
            nodes=[

                # =================================================
                # CONV1
                # =================================================
                Nested(
                    nodes=[

                        # -----------------------------------------
                        # Keep activation/output tile in AO
                        # -----------------------------------------
                        Storage(
                            component="AO",
                            tensors=["A", "O1"],
                        ),

                        # Weights stay in W SRAM
                        Storage(
                            component="W",
                            tensors=["W1"],
                        ),

                        # -----------------------------------------
                        # TEMPORAL LOOP ORDER
                        #
                        # for n
                        # for y1
                        # for x1 chunk
                        # for z1 output-feature block
                        # -----------------------------------------

                        Temporal(
                            rank_variable="n",
                            tile_shape=1,
                        ),

                        Temporal(
                            rank_variable="y1",
                            tile_shape=1,
                        ),

                        # One horizontal hardware block.
                        Temporal(
                            rank_variable="x1",
                            tile_shape=MAC_SPAT_UNITS,
                        ),

                        # NOF_ARRAYS output features together.
                        Temporal(
                            rank_variable="z1",
                            tile_shape=NOF_ARRAYS,
                        ),

                        # -----------------------------------------
                        # Local buffers
                        # -----------------------------------------

                        Storage(
                            component="ActivationReg",
                            tensors=["A"],
                        ),

                        Storage(
                            component="WeightReg",
                            tensors=["W1"],
                        ),

                        Storage(
                            component="OutputReg",
                            tensors=["O1"],
                        ),

                        # -----------------------------------------
                        # REDUCTION LOOP ORDER
                        #
                        # for input channel
                        #   for kernel row
                        #     for kernel col
                        # -----------------------------------------

                        Temporal(
                            rank_variable="z0",
                            tile_shape=1,
                        ),

                        Temporal(
                            rank_variable="ry",
                            tile_shape=1,
                        ),

                        Temporal(
                            rank_variable="rx",
                            tile_shape=1,
                        ),

                        # -----------------------------------------
                        # 240 X positions in parallel
                        # -----------------------------------------
                        Spatial(
                            rank_variable="x1",
                            tile_shape=1,
                            name="MACLane",
                            component="ComputeLanes",
                        ),

                        # -----------------------------------------
                        # MAC
                        # -----------------------------------------
                        Compute(
                            einsum="Conv1",
                            component="MAC",
                        ),
                    ],
                ),


                # =================================================
                # CONV2
                # =================================================
                Nested(
                    nodes=[

                        # O1 is now Conv2 input.
                        Storage(
                            component="AO",
                            tensors=["O1", "O2"],
                        ),

                        Storage(
                            component="W",
                            tensors=["W2"],
                        ),

                        # -----------------------------------------
                        # TEMPORAL LOOP ORDER
                        # -----------------------------------------

                        Temporal(
                            rank_variable="n",
                            tile_shape=1,
                        ),

                        Temporal(
                            rank_variable="y2",
                            tile_shape=1,
                        ),

                        Temporal(
                            rank_variable="x2",
                            tile_shape=MAC_SPAT_UNITS,
                        ),

                        Temporal(
                            rank_variable="z2",
                            tile_shape=NOF_ARRAYS,
                        ),

                        # -----------------------------------------
                        # Local buffers
                        # -----------------------------------------

                        Storage(
                            component="ActivationReg",
                            tensors=["O1"],
                        ),

                        Storage(
                            component="WeightReg",
                            tensors=["W2"],
                        ),

                        Storage(
                            component="OutputReg",
                            tensors=["O2"],
                        ),

                        # -----------------------------------------
                        # REDUCTION
                        #
                        # input channel -> kernel Y -> kernel X
                        # -----------------------------------------

                        Temporal(
                            rank_variable="z1",
                            tile_shape=1,
                        ),

                        Temporal(
                            rank_variable="ry2",
                            tile_shape=1,
                        ),

                        Temporal(
                            rank_variable="rx2",
                            tile_shape=1,
                        ),

                        # -----------------------------------------
                        # 240 output columns simultaneously
                        # -----------------------------------------
                        Spatial(
                            rank_variable="x2",
                            tile_shape=1,
                            name="MACLane",
                            component="ComputeLanes",
                        ),

                        Compute(
                            einsum="Conv2",
                            component="MAC",
                        ),
                    ],
                ),
            ],
        ),
    ],
)
spec = Spec(
    arch=arch,
    workload=workload,
    mapping=hand_mapping,
)

from accelforge import Spec, Metrics

# spec.mapper.metrics = Metrics.LATENCY
# spec.mapper.info_metrics = Metrics.all_metrics()

# spec.mapper.explore_imperfect_spatial_loops = True
# spec.mapper.explore_imperfect_temporal_loops = True

# # More freedom for Conv1/Conv2 fusion
# spec.mapper.max_fused_loops_per_rank_variable = 2
# spec.mapper.max_fused_loops = 4

# spec.mapper.prioritize_reuse_of_unfused_tensors = False

# spec.mapper.tiling_coarseness = 1.5
# spec.mapper.max_pmapping_templates_per_einsum = 10
# =========================================================
# RUN
# =========================================================

results = spec.map_workload_to_arch(
    print_progress=True,
    print_number_of_pmappings=True,
)

print(results)

In [ ]:
from accelforge.util._setexpressions import InvertibleSet


def repair_invertible_sets(obj, seen=None):
    if seen is None:
        seen = set()

    if id(obj) in seen:
        return
    seen.add(id(obj))

    if isinstance(obj, InvertibleSet):
        private = getattr(obj, "__pydantic_private__", None)

        if private is None:
            private = {}
            object.__setattr__(obj, "__pydantic_private__", private)

        private.setdefault("_bits_per_value", None)

    if isinstance(obj, dict):
        children = list(obj.keys()) + list(obj.values())
    elif isinstance(obj, (list, tuple, set, frozenset)):
        children = obj
    elif hasattr(obj, "__dict__"):
        children = obj.__dict__.values()
    else:
        children = ()

    for child in children:
        repair_invertible_sets(child, seen)

In [ ]:
results

In [ ]:
mapping = results.mapping()

repair_invertible_sets(mapping)

yaml_text = mapping.to_yaml()
print(yaml_text)